In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- STEP 1: Create the Mentor Dataset (mentors.csv) ---
mentor_data = {
    'Mentor_ID': ['M001', 'M002', 'M003', 'M004', 'M005'],
    'Name': ['Dr. Aris', 'Dr.Sarah', 'Coach Kevin', 'Prof. Elena', 'Dr. Miller'],
    'Expertise': ['Academic', 'Wellness', 'Career', 'Academic', 'Wellness'],
    'Capacity': [15, 12, 20, 15, 10],
    'Current_Load': [2, 5, 8, 4, 1]
}
mentors_df = pd.DataFrame(mentor_data)
mentors_df.to_csv('mentors.csv', index=False)
print(" mentors.csv created.")

# --- STEP 2: Load Student Data & Perform Clustering (Activity 3 Logic) ---
df = pd.read_csv('/content/student_scores (1).csv')
features = ['APS', 'WWS', 'PTMS', 'CRS']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# Clustering students into 4 groups
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# --- STEP 3: Mentor Matching Logic ---
def get_intervention(cluster):
    # Mapping based on the cluster characteristics found in your data
    mapping = {
        0: ("Career", "Career Roadmap & Industry Networking"),
        1: ("Wellness", "Urgent Wellness Check-in & Stress Management"),
        2: ("Academic", "Intensive Remedial Tutoring & Planning"),
        3: ("Career", "Leadership Mentoring & Placement Prep")
    }
    return mapping.get(cluster, ("General", "Standard Mentoring"))

def match_mentor(student_row, mentor_df):
    need, strategy = get_intervention(student_row['Cluster'])

    # Filter mentors by expertise and availability
    eligible = mentor_df[(mentor_df['Expertise'] == need) &
                         (mentor_df['Current_Load'] < mentor_df['Capacity'])]

    if not eligible.empty:
        # Load Balancing: Match with the mentor who has the most capacity left
        best_mentor = eligible.loc[(eligible['Capacity'] - eligible['Current_Load']).idxmax()]
        return best_mentor['Name'], need, strategy
    return "Queue", need, strategy

# Execute Matching
results = df.apply(lambda x: match_mentor(x, mentors_df), axis=1)
df[['Matched_Mentor', 'Need', 'Intervention']] = pd.DataFrame(results.tolist(), index=df.index)

# Simulate Alerts for High-Risk (Cluster 1 and 2)
df['Alert'] = df['Cluster'].apply(lambda x: ' CRITICAL' if x in [1, 2] else 'Normal')

# --- STEP 4: Save Final Table ---
df.to_csv('final_recommendations.csv', index=False)
print("final_recommendations.csv generated with 60 matches.")

 mentors.csv created.
final_recommendations.csv generated with 60 matches.
